# Document Extraction Engine: Kaggle demo

Upload a document and the engine will:

1. **Split** it into page images at 200 DPI.
2. **Classify** the document from page 1: category (structured, unstructured or handwritten), document type, language and rotation.
3. **Extract** the fields from each page with the model for that category.
4. **Draw** a box on each value, and build one JSON result.

**Before you start:** in the right panel, open **Session options** and set **Internet: ON** and **Accelerator: GPU T4 x2**. Then run the one code cell. Use synthetic (made-up) documents: the app link is public. Full instructions are in `docs/kaggle/README.md`.

## How the pipeline works

**1. Split the upload into page images.** A PDF is rendered at 200 DPI, one image per page. An image file is used as it is. Each page is cleaned up (contrast, deskew).

**2. Classify the document once, from page 1.** A small copy of page 1 goes to the router model (Qwen3-VL-4B). It returns:
- **category:** structured, unstructured or handwritten
- **document type:** DOC-01, DOC-02 or DOC-03 when a schema matches, otherwise generic
- **sub-type** for unstructured documents (brochure, pamphlet, menu card, and so on)
- **language** and **rotation**. Upright pages are turned upright before extraction.

The same result is applied to every page. You can override any of these by hand.

**3. Route each page by category.**

| Category | What reads the page | Boxes? |
| --- | --- | --- |
| Handwritten (including hand-filled forms) | Qwen3-VL-4B reads the page image | Yes |
| Structured, digital (typed or printed) | GLM-OCR reads the text, then Qwen3-4B-Instruct turns the text into JSON | No, values are flagged for review |
| Unstructured, digital | Same as structured, digital | No |

**4. Extract the fields.** The model returns each field's value and, where it can, a box as `[x1, y1, x2, y2]` on a 0–1000 scale. The app converts the box to pixels on the page. A fixed schema (DOC-01 to DOC-03) is used only when you pick the type. Otherwise the fields are named from the labels on the page.

**5. Check and merge.** Values for schema fields are normalised and checked (PAN, dates, amounts). Every page's fields are merged into one JSON with page numbers. Values without a box, failed checks and handwriting are flagged `review_required`.

**6. Build the JSON and draw the boxes.** The result has the document metadata, one entry per field with its value and evidence, the flags, and a record of which models ran. Each page is drawn with its boxes in the **Visual Grounding** tab.

**What is not yet measured:** accuracy on your documents, and how well the GLM-OCR route handles digital forms. Confidence is not calibrated, so every document is marked `needs_review`.


## Run this one cell

It gets the latest code, installs what is missing, starts three **vLLM** model servers inside this session, checks them with a synthetic page, and launches the app.

The first run takes a long time (installing vLLM and downloading about 20 GB). **vLLM has not yet been tried on Kaggle with this code.** If a step fails, copy the output and send it. To run the old way, without vLLM, set `USE_VLLM = False` at the top of the cell.

In [ ]:
# ONE CELL: get the latest code, install what is missing, start vLLM, check it, launch the app.
import importlib.util, os, subprocess, urllib.request

USE_VLLM = True   # False: load the models inside the app with Transformers (no vLLM)
URL = "https://github.com/Devadn11206/DOC_Extractor_QUAPT.git"
BRANCH = "fix/qwen3-only-and-validation-bugs"
REPO = "/kaggle/working/DOC_Extractor_QUAPT_vllm"
VLLM_ENV = "/kaggle/tmp/vllm-env"          # vLLM has its own environment so it cannot break the app's packages
LOGS = "/kaggle/working/vllm-logs"
os.chdir("/kaggle/working")


def sh(*cmd):
    return subprocess.run(list(cmd), capture_output=True, text=True)


def run(command):
    """Runs a shell command with live output and stops the cell if it fails."""
    get_ipython().system(command)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise SystemExit(f"Step failed, see the output above: {command}")


# 1. Code
if not os.path.isdir(REPO + "/.git"):
    r = sh("git", "clone", "-b", BRANCH, URL, REPO)
    if r.returncode != 0:
        raise SystemExit("Clone failed: turn Internet ON in the session options and re-run. " + r.stderr)
else:
    sh("git", "-C", REPO, "fetch", "origin", BRANCH)
    sh("git", "-C", REPO, "checkout", "-B", BRANCH, "origin/" + BRANCH)
print("Running commit:", sh("git", "-C", REPO, "log", "-1", "--oneline").stdout)

# 2. App packages (first run only)
needed = ["transformers", "qwen_vl_utils", "gradio", "bitsandbytes", "cv2", "pypdfium2", "instructor", "litellm"]
missing = [m for m in needed if importlib.util.find_spec(m) is None]
run("pip install -q uv")
if missing:
    print("Installing app dependencies, missing:", missing)
    run(f"cd {REPO} && uv pip install --system -r requirements.txt && uv pip install --system -e .")
    run("uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio")

os.environ.setdefault("DOC_MAX_IMAGE_PIXELS", "1300000")
os.chdir(REPO)

if USE_VLLM:
    # 3. vLLM (first run only, several minutes)
    if not os.path.exists(VLLM_ENV + "/bin/vllm"):
        run(f"uv venv --python 3.12 {VLLM_ENV}")
        run(f"uv pip install -p {VLLM_ENV}/bin/python --torch-backend=auto vllm bitsandbytes")

    # 4. Servers (the first start downloads about 20 GB). Running servers are reused.
    def ready(port):
        try:
            return urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5).status == 200
        except OSError:
            return False

    os.environ["VLLM_NO_USAGE_STATS"] = "1"
    if not all(ready(port) for port in (8000, 8001, 8002)):
        run(f"python scripts/serve_vllm.py --stop --log-dir {LOGS}")
        run(f"python scripts/serve_vllm.py --detach --log-dir {LOGS} --vllm-bin {VLLM_ENV}/bin/vllm")

    # 5. Check all three with a synthetic page
    os.environ["VLLM_API_BASE"] = "http://127.0.0.1:8000/v1"
    os.environ["VLLM_TEXT_API_BASE"] = "http://127.0.0.1:8001/v1"
    os.environ["VLLM_OCR_API_BASE"] = "http://127.0.0.1:8002/v1"
    os.environ["VLLM_TIMEOUT_SECONDS"] = "300"      # a T4 can be slow on a long page
    run("python scripts/check_vllm.py")
else:
    for name in ("VLLM_API_BASE", "VLLM_TEXT_API_BASE", "VLLM_OCR_API_BASE"):
        os.environ.pop(name, None)

# 6. App (public gradio.live link). Stop the cell to stop the app.
subprocess.run("fuser -k 7860/tcp 2>/dev/null || true", shell=True)
os.chdir(REPO + "/src")
get_ipython().system("python -m doc_extractor.app")

## Using the app

1. Upload a PDF, JPG, PNG or TIFF.
2. Leave the overrides on **Auto-Detect** to classify automatically.
3. Click **Classify & Extract** and watch the **Pipeline steps** panel.
4. Read the **Visual Grounding** tab (boxes on each page) and the **Canonical JSON Output** tab (download available).

**Expected behaviour**
- The first classification is slower, because the model loads then.
- Every page of a PDF is read, one page at a time.
- The digital route (GLM-OCR + text model) gives no boxes, so its values are withheld: `value: null`, `null_reason: EVIDENCE_MISSING`.
- Each extracted field has a `confidence` from the model's token probabilities; below 0.75 it is marked for review.

## Troubleshooting

| Problem | Fix |
| --- | --- |
| "Could not resolve host" | Turn Internet ON, restart the session, re-run the cell. |
| A vLLM server did not start | The end of its log is printed (full logs: `/kaggle/working/vllm-logs/`). See `docs/kaggle/vllm.md`. |
| `4/4 checks passed` does not appear | Send the lines that start with `FAIL`. |
| The app must work now and vLLM does not | Set `USE_VLLM = False`, restart the session, re-run the cell. |

Send the status line and the commit printed by the cell when asking for help.